# Notebook 7 – Encoding Categorical Variables



Categorical encoding converts category labels into numeric representations that a machine-learning algorithm can use. The best method depends on whether categories are nominal, ordinal, binary, or high-cardinality.

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder

## 1. Types of categorical variables
- **Nominal:** categories have no natural order (city, department, colour).
- **Ordinal:** categories have a meaningful order (low, medium, high).
- **Binary:** exactly two meaningful states (yes/no).
- **High cardinality:** a column has many distinct categories (postcode, product ID, user ID).

Do not assign arbitrary numeric ranks to nominal categories: a model may interpret the numbers as having order or distance.

In [ ]:
df = pd.DataFrame({
    "city": ["Chennai", "Salem", "Madurai", "Chennai", "Salem"],
    "satisfaction": ["Low", "High", "Medium", "High", "Low"],
    "subscribed": ["Yes", "No", "Yes", "No", "Yes"],
    "department": ["Sales", "IT", "HR", "IT", "Sales"],
    "purchased": [1, 0, 1, 1, 0]
})
df

## 2. Label encoding
`LabelEncoder` is generally designed for target labels (`y`), not nominal input features. It maps labels to integer IDs. Those IDs should not be interpreted as meaningful order.

In [ ]:
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(df["subscribed"])
pd.DataFrame({"original_target": df["subscribed"], "encoded_target": y_encoded})

## 3. Ordinal encoding
Use `OrdinalEncoder` when categories have a genuine order and define that order explicitly.

In [ ]:
order = [["Low", "Medium", "High"]]
ordinal_encoder = OrdinalEncoder(categories=order)
df["satisfaction_encoded"] = ordinal_encoder.fit_transform(df[["satisfaction"]]).astype(int)
df[["satisfaction", "satisfaction_encoded"]]

## 4. One-hot encoding
One-hot encoding creates a binary column for each category. It is usually suitable for nominal variables with a manageable number of categories.

In [ ]:
one_hot = pd.get_dummies(df["city"], prefix="city", dtype=int)
one_hot

## 5. Dummy variables
Dummy coding often drops one category to create a reference group, commonly when fitting models with an intercept to avoid perfect multicollinearity.

In [ ]:
dummy_vars = pd.get_dummies(df["city"], prefix="city", drop_first=True, dtype=int)
dummy_vars

## 6. Binary encoding
For two-category features, map values explicitly to 0/1 if that mapping is meaningful and documented.

In [ ]:
binary_map = {"No": 0, "Yes": 1}
df["subscribed_encoded"] = df["subscribed"].map(binary_map)
df[["subscribed", "subscribed_encoded"]]

## 7. Frequency encoding
Frequency encoding replaces each category with its count or proportion in the training data. It can reduce dimensionality, but different categories may receive the same value and frequency is not a semantic ranking.

In [ ]:
frequency_map = df["department"].value_counts(normalize=True)
df["department_frequency"] = df["department"].map(frequency_map)
df[["department", "department_frequency"]]

## 8. Target encoding
Target encoding replaces a category with a statistic of the target (for example, mean purchase rate). It can be powerful for high-cardinality features but has a serious leakage/overfitting risk.

**Rule:** calculate target statistics using training data only; for robust modeling, use out-of-fold target encoding and smoothing. Never compute target encodings using the full dataset before splitting.

In [ ]:
# Simple illustration only; do not use this naive in-sample encoding for model evaluation.
target_means = df.groupby("department")["purchased"].mean()
df["department_target_mean_demo"] = df["department"].map(target_means)
df[["department", "purchased", "department_target_mean_demo"]]

## 9. Handling unknown categories
At inference time, new categories may appear. Configure the encoder to handle unknowns or define a documented fallback strategy.

In [ ]:
# OneHotEncoder example with unknown-category handling
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded_array = encoder.fit_transform(df[["city"]])
encoded_df = pd.DataFrame(encoded_array, columns=encoder.get_feature_names_out(["city"]))
encoded_df

## 10. High-cardinality categories
Possible approaches include:
- One-hot encoding if the number of categories is manageable.
- Grouping genuinely rare categories into an `Other` bucket, based on training data and a clear policy.
- Frequency encoding.
- Target encoding with leakage-safe cross-fitting and smoothing.
- Domain-based features or embeddings for suitable models.

Avoid encoding unique IDs as if they were meaningful categories unless the ID itself contains learnable information.

## 11. Choosing a technique

| Technique | Best use | Main risk |
|---|---|---|
| Label encoding | Target labels; some tree workflows with care | False order for nominal input |
| Ordinal encoding | Ordered categories | Incorrect or undocumented order |
| One-hot encoding | Nominal features, low/moderate cardinality | Many columns for high cardinality |
| Dummy variables | Reference-category linear models | Reference interpretation; dropped category |
| Binary mapping | Two-category feature | Inconsistent mapping |
| Frequency encoding | High-cardinality categories | Same frequency can hide different categories |
| Target encoding | High-cardinality categories with target signal | Leakage and overfitting |

